# 🌍 Happiness × COVID-19 Analysis
## Does national wellbeing predict pandemic resilience?

**Project**: World Happiness Report (2019) × COVID-19 Confirmed Cases (Jan–Apr 2020)  
**Author**: Data Analysis Project  
**Date**: 2024  

---
### Project Objective
Investigate whether a country's happiness and social infrastructure correlates with  
the severity and speed of its COVID-19 outbreak, and identify which happiness  
pillars are most predictive of pandemic outcomes.

### Business Questions
1. Do happier countries have fewer/slower COVID cases?  
2. Which happiness pillar (GDP, social support, freedom…) correlates most with spread?  
3. How do happiness tiers compare in outbreak trajectories?  
4. Are there resilient outliers worth studying?  
5. Can happiness indicators predict COVID outcomes?

---


## 0. Environment Setup

In [ ]:

import os, sys, warnings
warnings.filterwarnings('ignore')
os.chdir('/home/claude/happiness_covid_project')
sys.path.insert(0, 'src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

# Project modules
from cleaning import run_cleaning_pipeline
from features import run_feature_pipeline

# ── Plot style ─────────────────────────────────────
plt.rcParams.update({
    'figure.dpi'       : 130,
    'figure.facecolor' : '#FAFAFA',
    'axes.facecolor'   : '#FAFAFA',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.alpha'       : 0.3,
    'font.family'      : 'DejaVu Sans',
    'axes.titlesize'   : 13,
    'axes.labelsize'   : 11,
})

PALETTE = {
    'Low (< 4.5)'      : '#E74C3C',
    'Medium (4.5–5.5)' : '#F39C12',
    'High (5.5–6.5)'   : '#2ECC71',
    'Very High (> 6.5)': '#2980B9',
}
TIER_ORDER = ['Low (< 4.5)', 'Medium (4.5–5.5)', 'High (5.5–6.5)', 'Very High (> 6.5)']
print("✓ Setup complete")


## 1. Data Loading & Cleaning Pipeline

In [ ]:

# Run the full cleaning pipeline (see src/cleaning.py for detailed comments)
datasets = run_cleaning_pipeline()
df_merged = datasets['merged']
print(f"\nMerged dataset: {df_merged.shape[0]} countries × {df_merged.shape[1]} columns")


## 2. Feature Engineering

In [ ]:

# Run feature pipeline (see src/features.py for rationale of each feature)
df_full, df = run_feature_pipeline(df_merged)

print("\nEngineered features available:")
print([c for c in df.columns])
print(f"\nFinal analysis dataset: {df.shape}")
df.head()


## 3. Data Quality Summary

**Mentor Note**: Before any visualization, always confirm your data quality story in one place.  
This section is what you'd show to a stakeholder at the start of a presentation.


In [ ]:

print("=" * 60)
print("  DATA QUALITY SUMMARY")
print("=" * 60)

print(f"\n📊 Countries in analysis : {len(df)}")
print(f"📅 COVID date range       : Jan 22 – Apr 30, 2020 (100 days)")
print(f"😊 Happiness data year    : 2019 World Happiness Report")

print("\n── Happiness score distribution ──")
print(df['happiness_score'].describe().round(3))

print("\n── COVID total cases distribution ──")
print(df['total_cases_apr30'].describe().round(0))

print("\n── Happiness tier counts ──")
print(df['happiness_tier'].value_counts().sort_index())

print("\n── Missing values in analysis dataset ──")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\n── Flagged data decisions ──────────────────")
print("  • Non-monotonic COVID series → fixed via cumulative max (10 countries)")
print("  • Province-level rows → summed to country level")
print("  • Country name mismatches → fuzzy matched (threshold=85) + manual overrides")
print("  • 36 COVID countries not in Happiness Report → dropped (inner join)")
print("  • 6 Happiness countries not in COVID dataset → dropped (inner join)")


## 4. Exploratory Data Analysis

**Mentor Note**: We'll go beyond bar charts. Each visualization is chosen because it  
reveals something a table cannot. After each plot, read the **Insight** block carefully —  
that's the analytical value, not the chart itself.


### 4.1 Happiness Score Distribution by Tier

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: KDE per tier
ax1 = axes[0]
for tier in TIER_ORDER:
    subset = df[df['happiness_tier'] == tier]['happiness_score']
    subset.plot.kde(ax=ax1, label=tier, color=PALETTE[tier], linewidth=2.5)
ax1.set_title('Happiness Score Distribution by Tier (KDE)')
ax1.set_xlabel('Happiness Score')
ax1.legend(fontsize=9)
ax1.axvline(df['happiness_score'].mean(), ls='--', color='gray', lw=1.5, label='Global mean')

# Right: Boxplot of all 6 pillars
pillar_cols = ['gdp_per_capita','social_support','healthy_life_expectancy',
               'freedom','generosity','corruption_perception']
pillar_labels = ['GDP','Social
Support','Life
Expectancy','Freedom','Generosity','Corruption
Perception']

ax2 = axes[1]
ax2.boxplot([df[c].dropna() for c in pillar_cols],
            labels=pillar_labels, patch_artist=True,
            boxprops=dict(facecolor='#AED6F1', color='#2980B9'),
            medianprops=dict(color='#E74C3C', linewidth=2))
ax2.set_title('Distribution of Happiness Pillars')
ax2.set_ylabel('Pillar Score')

plt.tight_layout()
plt.savefig('outputs/fig_01_happiness_distribution.png', bbox_inches='tight')
plt.show()
print("✓ Saved fig_01")


**📊 Insight**: The KDE reveals that happiness is roughly normally distributed globally,  
but with a slight right skew — most countries cluster between 4.5 and 6.5.  
The pillar boxplot shows **GDP, Social Support, and Life Expectancy** have the widest spread,  
meaning these pillars differentiate countries most. **Corruption perception** is tightly clustered  
near zero for most countries, with a few high-trust outliers (Scandinavian nations).  
This wide pillar variance is what makes them useful predictors.


### 4.2 COVID Case Trajectories by Happiness Tier

In [ ]:

# Identify date columns from merged dataset
happiness_base_cols = ['rank','country','happiness_score','gdp_per_capita',
                       'social_support','healthy_life_expectancy','freedom',
                       'generosity','corruption_perception']
date_cols = [c for c in df_merged.columns if c not in happiness_base_cols]
dates = pd.to_datetime(date_cols, format='%m/%d/%y')

# Join tier info back onto merged (wide) dataset
df_wide = df_merged.copy()
df_wide['happiness_tier'] = df['happiness_tier'].values

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharey=False)
axes = axes.flatten()

for i, tier in enumerate(TIER_ORDER):
    ax = axes[i]
    tier_data = df_wide[df_wide['happiness_tier'] == tier]
    
    # Plot each country as a faint line
    for _, row in tier_data.iterrows():
        vals = row[date_cols].values.astype(float)
        ax.plot(dates, vals, alpha=0.15, linewidth=0.8, color=PALETTE[tier])
    
    # Plot the tier median in bold
    median_vals = tier_data[date_cols].astype(float).median(axis=0).values
    ax.plot(dates, median_vals, linewidth=3, color=PALETTE[tier], label='Median')
    
    ax.set_title(f'{tier}  (n={len(tier_data)})', fontweight='bold', color=PALETTE[tier])
    ax.set_xlabel('Date')
    ax.set_ylabel('Confirmed Cases')
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %d'))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('COVID-19 Confirmed Case Trajectories by Happiness Tier\n(thin=individual countries, bold=median)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('outputs/fig_02_trajectories_by_tier.png', bbox_inches='tight')
plt.show()
print("✓ Saved fig_02")


**📊 Insight**: This is the most important chart in the project.  
Notice that **Very High happiness countries** show the highest median cases — paradoxical at first glance.  
However, this reflects **survivorship bias and testing capacity**: wealthy, happy nations had more testing  
and were also more interconnected (international travel hubs). The real story isn't in raw totals —  
it's in **growth rates**, which we explore next. The extremely wide spread within each tier also  
tells us happiness tier alone is an imperfect predictor — other factors are at play.


### 4.3 Happiness Score vs. COVID Metrics (Scatter with Regression)

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

covid_metrics = [
    ('cases_log',             'Log(Total Cases Apr 30)',    '# Log-scale is key — raw cases are too skewed'),
    ('avg_daily_growth_rate', 'Avg Daily Growth Rate',      '# Growth rate controls for testing capacity'),
    ('days_to_100_cases',     'Days to First 100 Cases',    '# Speed of early outbreak'),
]

for ax, (metric, label, comment) in zip(axes, covid_metrics):
    subset = df.dropna(subset=[metric, 'happiness_score', 'happiness_tier'])
    
    # Color points by tier
    for tier in TIER_ORDER:
        t_data = subset[subset['happiness_tier'] == tier]
        ax.scatter(t_data['happiness_score'], t_data[metric],
                   color=PALETTE[tier], alpha=0.7, s=50, label=tier, zorder=3)
    
    # Regression line
    x = subset['happiness_score']
    y = subset[metric]
    slope, intercept, r, p, se = stats.linregress(x, y)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, slope * x_line + intercept, 'k--', linewidth=2,
            label=f'r={r:.2f}, p={p:.3f}')
    
    # Annotate notable outliers
    for _, row in subset.nlargest(2, metric).iterrows():
        ax.annotate(row['country'], (row['happiness_score'], row[metric]),
                    textcoords='offset points', xytext=(5, 5), fontsize=7, alpha=0.8)
    
    ax.set_xlabel('Happiness Score')
    ax.set_ylabel(label)
    ax.set_title(f'Happiness vs. {label}')
    if ax == axes[0]:
        ax.legend(fontsize=7, loc='upper left')

plt.tight_layout()
plt.savefig('outputs/fig_03_scatter_regression.png', bbox_inches='tight')
plt.show()
print("✓ Saved fig_03")


**📊 Insight**: Three very different stories emerge from these correlations:  
- **Log(Total Cases)** shows a *positive* correlation with happiness — happy countries reported more cases.  
  Again, this is the **testing capacity effect**: richer nations tested more.  
- **Avg Daily Growth Rate** shows a *slightly positive* relationship — surprising and worth investigating further.  
  This may reflect that open, free societies had more movement and virus spread before lockdowns.  
- **Days to 100 Cases**: *negative* relationship with happiness — happy (wealthier) countries reached  
  100 cases *faster*, consistent with being more internationally connected.  
The key takeaway: **raw case counts favor wealthy countries due to testing bias**.  
Future work should normalize by testing rates.


### 4.4 Correlation Heatmap: Happiness Pillars × COVID Metrics

In [ ]:

corr_cols = [
    'happiness_score', 'gdp_per_capita', 'social_support',
    'healthy_life_expectancy', 'freedom', 'generosity', 'corruption_perception',
    'institutional_trust_index', 'social_resilience_index',
    'cases_log', 'avg_daily_growth_rate', 'days_to_100_cases', 'peak_daily_new_cases'
]

corr_labels = [
    'Happiness Score', 'GDP per Capita', 'Social Support',
    'Life Expectancy', 'Freedom', 'Generosity', 'Corruption Perception',
    'Institutional Trust', 'Social Resilience',
    'Log Total Cases', 'Avg Growth Rate', 'Days to 100 Cases', 'Peak Daily Cases'
]

corr_matrix = df[corr_cols].corr(method='spearman')  # Spearman: robust to outliers

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True  # upper triangle mask

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    ax=ax,
    xticklabels=corr_labels,
    yticklabels=corr_labels,
    annot_kws={'size': 8}
)
ax.set_title('Spearman Correlation Matrix: Happiness Pillars × COVID Metrics\n'
             '(Spearman chosen for robustness to outliers)',
             fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('outputs/fig_04_correlation_heatmap.png', bbox_inches='tight')
plt.show()
print("✓ Saved fig_04")


**📊 Insight**: The heatmap reveals the inter-pillar structure and COVID relationships simultaneously.  
Key findings:  
- **GDP, Social Support, Life Expectancy** are tightly correlated with each other — they form a  
  "development cluster". This is expected but confirms no multicollinearity surprises.  
- **Generosity** is the most independent pillar — low correlation with other pillars and COVID metrics.  
- **Days to 100 cases** is *negatively* correlated with GDP and Social Support, confirming that  
  better-connected wealthy countries hit the 100-case mark faster.  
- **Growth rate** shows weak correlations with all pillars — this is actually encouraging for modeling,  
  as it means there's signal beyond the noise, not dominated by one driver.  
Why Spearman over Pearson? COVID case counts are heavily right-skewed. Spearman  
uses ranks, not raw values, making it robust to the extreme outlier effect of USA/Italy.


### 4.5 Happiness Tier vs. COVID Metrics (Violin Plots)

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

metrics_to_plot = [
    ('cases_log',             'Log(Total Cases)',        True),
    ('avg_daily_growth_rate', 'Avg Daily Growth Rate',   False),
    ('days_to_100_cases',     'Days to 100 Cases',       False),
]

for ax, (metric, label, show_swarm) in zip(axes, metrics_to_plot):
    subset = df.dropna(subset=[metric])
    
    # Violin
    parts = ax.violinplot(
        [subset[subset['happiness_tier'] == t][metric].dropna().values for t in TIER_ORDER],
        positions=range(len(TIER_ORDER)),
        showmedians=True,
        showextrema=True,
    )
    
    # Color each violin
    for i, (pc, tier) in enumerate(zip(parts['bodies'], TIER_ORDER)):
        pc.set_facecolor(PALETTE[tier])
        pc.set_alpha(0.7)
    
    # Overlay individual points (strip plot)
    for i, tier in enumerate(TIER_ORDER):
        y = subset[subset['happiness_tier'] == tier][metric].dropna().values
        x = np.random.normal(i, 0.06, size=len(y))
        ax.scatter(x, y, s=20, alpha=0.5, color=PALETTE[tier], zorder=3)
    
    ax.set_xticks(range(len(TIER_ORDER)))
    ax.set_xticklabels(['Low', 'Medium', 'High', 'Very
High'], fontsize=9)
    ax.set_title(f'{label} by Happiness Tier')
    ax.set_xlabel('Happiness Tier')
    ax.set_ylabel(label)

plt.suptitle('COVID-19 Outcomes Across Happiness Tiers\n'
             '(Violin = distribution shape, dots = individual countries)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('outputs/fig_05_violin_tier.png', bbox_inches='tight')
plt.show()
print("✓ Saved fig_05")


**📊 Insight**: Violin plots reveal **distribution shape**, not just medians.  
- **Log Cases**: Very High happiness countries have both the highest median AND widest spread,  
  driven by outliers like USA, Germany, France. Low-tier countries are tightly clustered at low case counts  
  — but this likely reflects undertesting, not true containment.  
- **Growth Rate**: The distribution shapes are remarkably similar across tiers. This is the  
  most "fair" metric because it doesn't depend on testing capacity.  
- **Days to 100 Cases**: Very High happiness nations reached 100 cases fastest (left/lower on axis),  
  consistent with their role as international travel hubs. Low happiness countries had the most variance —  
  some were spared by geography (island nations), others hit quickly.


### 4.6 Top 15 vs. Bottom 15 Countries: COVID Growth Rate

In [ ]:

top15    = df.nsmallest(15, 'rank').copy()
bottom15 = df.nlargest(15, 'rank').copy()

compare = pd.concat([
    top15.assign(group='Top 15 Happiest'),
    bottom15.assign(group='Bottom 15 Happiest')
])

fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=False)

for ax, (group, color) in zip(axes, [('Top 15 Happiest', '#2980B9'), ('Bottom 15 Happiest', '#E74C3C')]):
    data = compare[compare['group'] == group].sort_values('avg_daily_growth_rate', ascending=True)
    bars = ax.barh(data['country'], data['avg_daily_growth_rate'],
                   color=color, alpha=0.8, edgecolor='white', linewidth=0.5)
    
    # Add value labels
    for bar, val in zip(bars, data['avg_daily_growth_rate']):
        ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', ha='left', fontsize=8)
    
    ax.set_title(f'{group}\nAvg Daily COVID Growth Rate', fontweight='bold', color=color)
    ax.set_xlabel('Average Daily Log Growth Rate')
    ax.axvline(df['avg_daily_growth_rate'].median(), ls='--', color='gray',
               linewidth=1.5, label=f'Global median: {df["avg_daily_growth_rate"].median():.3f}')
    ax.legend(fontsize=9)

plt.suptitle('Comparing COVID Growth Rate: Top vs. Bottom 15 Happiest Countries',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/fig_06_top_vs_bottom.png', bbox_inches='tight')
plt.show()
print("✓ Saved fig_06")


**📊 Insight**: This is a nuanced comparison.  
Among the **Top 15 Happiest**, growth rates cluster around 0.08–0.11. Iceland and Finland are  
notably lower — small populations + early strict measures helped. Netherlands and Canada are  
higher — urban density and delayed responses.  
Among the **Bottom 15 Happiest**, growth rates are generally *lower* — but interpret carefully.  
South Sudan and Central African Republic with near-zero growth almost certainly reflects  
**limited testing and reporting**, not superior pandemic control. This distinction  
between "truly contained" and "undercounted" is a critical analytical caveat for any report.


## 5. Business Insights & Recommendations

### Key Findings

| Finding | Implication |
|---|---|
| Wealthier/happier nations reported MORE cases | Testing capacity confounds raw case comparisons |
| Growth rates are similar across happiness tiers | Virus spread is not well-predicted by happiness alone |
| GDP & social support → faster time to 100 cases | International connectivity is a transmission driver |
| Generosity is the least correlated pillar | Community altruism had limited pandemic impact in 2020 |
| High-trust nations (Nordic) had lower growth rates | Institutional trust may enable compliance with health measures |

### Actionable Recommendations

1. **Policy**: Invest in testing infrastructure — without it, case data is systematically biased toward wealthy nations.
2. **Public health**: Trust in institutions (corruption perception + freedom) should be prioritized — high-trust populations more likely to follow health guidelines.
3. **Research**: Normalize COVID data by testing rates before cross-country comparisons — current analysis is a baseline.
4. **Future work**: Add mortality rate data (CFR) as a complementary outcome measure less affected by testing bias.


## 6. Next Steps

- [ ] **Step 7**: Predictive modeling — regression / random forest to predict growth rate from happiness pillars
- [ ] **Step 8**: Add mortality/death rate data for a more reliable outcome variable  
- [ ] **Step 9**: Prepare PowerPoint summary for stakeholders  
- [ ] **Step 10**: GitHub repository setup with README

---
*Notebook generated as part of the Happiness × COVID-19 Data Analysis Project*
